# GamaX1 (Aetherion) — Colab GPU Training

**Setup order (important):**
1. `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** (or better if available) → Save.
2. Run cells top to bottom.

**Why Drive matters here:** Colab's free tier can disconnect a session (12h hard cap, idle timeout, or GPU pre-emption) with no warning. Checkpoints are written straight to Google Drive, and the training command auto-resumes from the latest checkpoint if one exists — so a disconnect costs you at most a few minutes of progress, not the whole run.

## 1. Confirm GPU is attached

In [8]:
!nvidia-smi

Sun Aug  2 07:18:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

If this errors out with "command not found" or shows no GPU, go back to `Runtime` → `Change runtime type` and select a GPU, then re-run this cell.

## 2. Mount Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All persistent project files live under this Drive folder.
PROJECT_ROOT = '/content/drive/MyDrive/Aetherion_GamaX1'
CORPUS_DIR   = f'{PROJECT_ROOT}/data'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints_bpe_5gb'

os.makedirs(CORPUS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Corpus dir  :', CORPUS_DIR)
print('Checkpoints :', CKPT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Aetherion_GamaX1
Corpus dir  : /content/drive/MyDrive/Aetherion_GamaX1/data
Checkpoints : /content/drive/MyDrive/Aetherion_GamaX1/checkpoints_bpe_5gb


## 3. Get the GamaX1 codebase onto the runtime

Pick **one** of the two options below depending on where your code currently lives.

**Option A — code already sitting in Drive** (e.g. you zipped/copied the `gamax1` project folder into Drive already):

In [11]:
import os
import shutil

# Locate the project by its package entrypoint instead of assuming a
# particular Drive folder name.
project_matches = []
for drive_root, _, drive_files in os.walk('/content/drive/MyDrive'):
    if os.path.isfile(os.path.join(drive_root, 'gamax1', 'train.py')):
        project_matches.append(drive_root)

if not project_matches:
    raise FileNotFoundError(
        'Could not find the GamaX1 project in Drive. Upload the project '
        'folder, or use Option B to clone it from GitHub.'
    )

CODE_SRC_IN_DRIVE = project_matches[0]
TARGET_PROJECT = '/content/gamax1_project'
if os.path.exists(TARGET_PROJECT):
    shutil.rmtree(TARGET_PROJECT)
shutil.copytree(CODE_SRC_IN_DRIVE, TARGET_PROJECT)
os.chdir(TARGET_PROJECT)
print('Using project:', CODE_SRC_IN_DRIVE)
print('Runtime project:', os.getcwd())
print(os.listdir(TARGET_PROJECT))

FileNotFoundError: Could not find the GamaX1 project in Drive. Upload the project folder, or use Option B to clone it from GitHub.

**Option B — code lives in a GitHub repo instead.** Skip the cell above and use this one (edit the URL):

In [ ]:
# %cd /content
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git gamax1_project
# %cd /content/gamax1_project
# !ls

## 4. Put your 5GB corpus in place

Upload your corpus `.txt` file into `CORPUS_DIR` in Drive beforehand (via the Drive web UI, or `rclone`/`gdown` if it's hosted elsewhere — uploading a 5GB file through the browser is slow but only needs to happen once). Then point `CORPUS_PATH` at it below.

In [12]:
import os
import subprocess

CORPUS_FILENAME = 'corpus_400_books.txt'
CORPUS_PATH = f'{CORPUS_DIR}/{CORPUS_FILENAME}'

# Find the corpus if it was uploaded somewhere else in Google Drive.
if not os.path.exists(CORPUS_PATH):
    for drive_root, _, drive_files in os.walk('/content/drive/MyDrive'):
        if CORPUS_FILENAME in drive_files:
            CORPUS_PATH = os.path.join(drive_root, CORPUS_FILENAME)
            break

# If it is not present, build it from the 400 Gutenberg IDs configured in
# prepare_large_corpus.py. This can take several minutes.
if not os.path.exists(CORPUS_PATH):
    prepare_candidates = [
        '/content/gamax1_project/prepare_large_corpus.py',
        f'{PROJECT_ROOT}/GamaX1_Aetherion_v1/gamax1/prepare_large_corpus.py',
        f'{PROJECT_ROOT}/prepare_large_corpus.py',
    ]
    prepare_script = next((p for p in prepare_candidates if os.path.exists(p)), None)
    if prepare_script is None:
        raise FileNotFoundError(
            f'{CORPUS_FILENAME} was not found. Upload it to {CORPUS_DIR}, '
            'or run the code setup cell first so the 400-book preparation script is available.'
        )
    print(f'Building corpus from 400 Gutenberg IDs using {prepare_script} ...')
    subprocess.run([
        'python', prepare_script, '--out', CORPUS_PATH, '--delay', '1'
    ], check=True)

size_gb = os.path.getsize(CORPUS_PATH) / (1024**3)
print(f'Found corpus: {CORPUS_PATH} ({size_gb:.2f} GB)')

FileNotFoundError: corpus_400_books.txt was not found. Upload it to /content/drive/MyDrive/Aetherion_GamaX1/data, or run the code setup cell first so the 400-book preparation script is available.

## 5. Install dependencies

In [ ]:
!pip install -q torch --extra-index-url https://download.pytorch.org/whl/cu121
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — check runtime type!')

## 6. Auto-resume training command

This cell finds the newest checkpoint in `CKPT_DIR` (if any) and passes it via `--resume_from`, so re-running this exact cell after any disconnect continues from where it left off instead of restarting. Model/optimizer/sparsity-controller state all restore correctly since the checkpoint stores all of it.

**Note on model size:** `--auto_size_model` picks from a small fixed set of architecture sizes and was tuned for smaller corpora. On a 5GB / ~1.25B-token corpus it will likely under-size the model relative to the data (you saw this tradeoff discussed already). Manual `--d_model` / `--n_layers` / `--n_features` values are set below as a reasonable starting point — replace them with whatever sizing we calculate together before a long run, rather than trusting the auto-sizer here.

In [ ]:
import glob, os

ckpts = sorted(
    glob.glob(f'{CKPT_DIR}/gamax1_step_*.pt'),
    key=lambda p: int(p.rsplit('_', 1)[1].split('.')[0])
)
resume_flag = f'--resume_from "{ckpts[-1]}"' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else '(no checkpoint found — starting fresh)')

# EDIT these once you've settled on a model size for the 5GB corpus.
D_MODEL    = 512
N_HEADS    = 8
N_LAYERS   = 6
N_FEATURES = 2048

train_cmd = f'''
python -m gamax1.train \\
  --tokenizer bpe \\
  --data "{CORPUS_PATH}" \\
  --d_model {D_MODEL} \\
  --n_heads {N_HEADS} \\
  --n_layers {N_LAYERS} \\
  --n_features {N_FEATURES} \\
  --bpe_vocab_size 8000 \\
  --max_steps 20000 \\
  --checkpoint_interval 500 \\
  --out_dir "{CKPT_DIR}" \\
  {resume_flag}
'''.strip()

print(train_cmd)

In [ ]:
!{train_cmd}

## 7. Generate a sample once training is done (or paused)

Always point `--ckpt` at the file inside `CKPT_DIR` (Drive), never a local Colab path — the local disk is wiped when the session ends.

In [ ]:
!python -m gamax1.generate --ckpt "{CKPT_DIR}/gamax1.pt" --prompt "Prince Andrew" --max_new_tokens 500

## Notes / gotchas

- **If the session disconnects mid-run:** just reconnect, re-run cells 1–5, then re-run cell 6 — it will detect the latest checkpoint in Drive and resume automatically.
- **Free tier session cap is ~12 hours**, and GPU availability isn't guaranteed at peak times. If you hit a wall, Colab Pro (~$10/month) gets priority access and sometimes better GPUs (L4/A100), or switch to RunPod/Vast.ai for guaranteed dedicated GPU time.
- **`--checkpoint_interval 500`** keeps checkpoint loss small on disconnect — lower it further (e.g. 250) if you're on a flaky connection, at the cost of a bit more Drive-write overhead.
- Drive writes can occasionally lag — if `!ls {CKPT_DIR}` doesn't show a checkpoint you just expect to see, wait a few seconds and re-check before assuming something failed.